# Import thư viện và dữ liệu

In [1]:
import pandas as pd
import os
import configparser
from neo4j import GraphDatabase
import textwrap

# Thiết lập định dạng hiển thị cho các giá trị float, tránh dùng ký hiệu khoa học cho số lớn
pd.options.display.float_format = '{:.0f}'.format

# Kết nối tới Neo4j

In [3]:
# Sử dụng file ini cho thông tin đăng nhập, nếu không thì cung cấp mặc định
HOST = 'neo4j://localhost'
USERNAME = 'neo4j'
DATABASE = 'neo4j'
PASSWORD = 'password'

NEO4J_CONF_FILE = 'neo4j.ini'

if NEO4J_CONF_FILE is not None and os.path.exists(NEO4J_CONF_FILE):
    config = configparser.RawConfigParser()
    config.read(NEO4J_CONF_FILE)
    HOST = config['NEO4J']['HOST']
    DATABASE = config['NEO4J'].get('DATABASE', 'neo4j')
    USERNAME = config['NEO4J'].get('USERNAME', DATABASE)
    PASSWORD = config['NEO4J']['PASSWORD']
    print('Using custom database properties')
else:
    print('Could not find database properties file, using defaults')

Using custom database properties


In [4]:
# Tạo Neo4j Python driver
driver = GraphDatabase.driver(HOST, auth=(USERNAME, PASSWORD))

In [5]:
# Hàm bổ trợ
def run(driver, query, params=None):
    with driver.session(database=DATABASE) as session:
        if params is not None:
            return [r for r in session.run(query, params)]
        else:
            return [r for r in session.run(query)]

## Xóa dữ liệu cũ (Clear Database)

In [6]:
# Xóa toàn bộ các node và quan hệ trong database trước khi load mới
# 1. Xóa toàn bộ các quan hệ (relationships) theo lô 500,000
print("Bắt đầu xóa các quan hệ...")
deleted_relations = True
while deleted_relations:
    result = run(driver, "MATCH ()-[r]->() WITH r LIMIT 500000 DELETE r RETURN count(r) as count")
    count = result[0]['count']
    print(f"Đã xóa {count} quan hệ...")
    deleted_relations = count > 0
# 2. Xóa toàn bộ các nodes theo lô 500,000
print("Bắt đầu xóa các nodes...")
deleted_nodes = True
while deleted_nodes:
    result = run(driver, "MATCH (n) WITH n LIMIT 500000 DELETE n RETURN count(n) as count")
    count = result[0]['count']
    print(f"Đã xóa {count} nodes...")
    deleted_nodes = count > 0
print("Đã dọn dẹp sạch sẽ database thành công!")

Bắt đầu xóa các quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 500000 quan hệ...
Đã xóa 370619 quan hệ...
Đã xóa 0 quan hệ...
Bắt đầu xóa các nodes...
Đã xóa 51870 nodes...
Đã xóa 0 nodes...
Đã dọn dẹp sạch sẽ database thành công!


## Thiết lập các ràng buộc duy nhất (Unique Constraints)

In [7]:
run(driver,'CREATE CONSTRAINT user_id_unique IF NOT EXISTS FOR (user:User) REQUIRE user.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT origin_id_unique IF NOT EXISTS FOR (origin:Origin) REQUIRE origin.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT poi_id_unique IF NOT EXISTS FOR (poi:Poi) REQUIRE poi.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT category_id_unique IF NOT EXISTS FOR (category:Category) REQUIRE category.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT region_id_unique IF NOT EXISTS FOR (region:Region) REQUIRE region.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT review_id_unique IF NOT EXISTS FOR (review:Review) REQUIRE review.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT ward_code_unique IF NOT EXISTS FOR (w:Ward) REQUIRE w.code IS UNIQUE')

run(driver,'CREATE CONSTRAINT district_id_unique IF NOT EXISTS FOR (d:District) REQUIRE d.id IS UNIQUE')

run(driver,'CREATE CONSTRAINT district_name_unique IF NOT EXISTS FOR (d:District) REQUIRE d.name IS UNIQUE')


[]

In [8]:
# Thiết lập thư mục chứa các file .csv đã tiền xử lý

def get_file_url(filename):
    return f"file:///{filename}"

url_node_category = get_file_url('df_node_category.csv')
url_node_origin = get_file_url('df_node_origin.csv')
url_node_poi = get_file_url('df_node_poi.csv')
url_node_region = get_file_url('df_node_region.csv')
url_node_review_1 = get_file_url('df_node_review_1.csv')
url_node_review_2 = get_file_url('df_node_review_2.csv')
url_node_user = get_file_url('df_node_user.csv')
url_poi_belongsto_category = get_file_url('df_poi_belongsto_category.csv')
url_poi_locatedat_region = get_file_url('df_poi_locatedat_region.csv')
url_user_from_origin = get_file_url('df_user_from_origin.csv')
url_user_reviewed_poi = get_file_url('df_user_reviewed_poi.csv')
# Ontology địa chỉ cũ & mới
url_node_district = get_file_url('df_node_district.csv')
url_node_ward = get_file_url('df_node_ward.csv')
url_poi_locatedin_ward = get_file_url('df_poi_locatedin_ward.csv')
url_ward_mergedto_ward = get_file_url('df_ward_mergedto_ward.csv')


## Tải các Node (Nút)

In [9]:
# df_node_category
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE(category:Category {id: toInteger(row.id), name: row.name})
    RETURN count(category)
    """),
    params = {'file': url_node_category}
)

[<Record count(category)=452>]

In [10]:
# df_node_origin
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE(origin:Origin {id: toInteger(row.id), name: row.name})
    RETURN count(origin)
    """),
    params = {'file': url_node_origin}
)

[<Record count(origin)=2343>]

In [11]:
#df_node_poi
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE (poi:Poi {id: toInteger(row.id)})
    ON CREATE SET
        poi.name = coalesce(row.name, ''),
        poi.description = coalesce(row.description, ''),
        poi.url = coalesce(row.url, ''),
        poi.openingHours = coalesce(row.openingHours, ''),
        poi.duration = coalesce(row.duration, ''),
        poi.price = toFloat(coalesce(row.price, '0.0')),
        poi.address = coalesce(row.address, ''),
        poi.avgRating = toFloat(coalesce(row.avgRating, '0.0')),
        poi.numReviews = toInteger(coalesce(row.numReviews, '0')),
        poi.numReviews_5 = toInteger(coalesce(row.numReviews_5, '0')),
        poi.numReviews_4 = toInteger(coalesce(row.numReviews_4, '0')),
        poi.numReviews_3 = toInteger(coalesce(row.numReviews_3, '0')),
        poi.numReviews_2 = toInteger(coalesce(row.numReviews_2, '0')),
        poi.numReviews_1 = toInteger(coalesce(row.numReviews_1, '0')),
        poi.latitude = case when coalesce(row.latitude, '') = '' then null else toFloat(row.latitude) end,
        poi.longitude = case when coalesce(row.longitude, '') = '' then null else toFloat(row.longitude) end,
        poi.location = case when coalesce(row.latitude, '') = '' or coalesce(row.longitude, '') = '' then null 
                            else point({latitude: toFloat(row.latitude), longitude: toFloat(row.longitude)}) end
    RETURN count(poi)
    """),
    params = {'file': url_node_poi}
)


[<Record count(poi)=3326>]

In [12]:
# df_node_region
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE(region:Region {id: toInteger(row.id), name: row.name})
    RETURN count(region)
    """),
    params = {'file': url_node_region}
)

[<Record count(region)=22>]

In [13]:
#df_node_review_1
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE (review:Review {id: toInteger(row.id)})
    ON CREATE SET
        review.title = coalesce(row.title, ''),
        review.date = case when coalesce(row.date, '') = '' then null else date(row.date) end,
				review.rating = toFloat(coalesce(row.rating, '0.0')),
				review.content = coalesce(row.content, '')
    RETURN count(review)
    """),
    params = {'file': url_node_review_1}
)

[<Record count(review)=12054>]

In [14]:
#df_node_review_2
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE (review:Review {id: toInteger(row.id)})
    ON CREATE SET
        review.title = coalesce(row.title, ''),
        review.date = case when coalesce(row.date, '') = '' then null else date(row.date) end,
				review.rating = toFloat(coalesce(row.rating, '0.0')),
				review.content = coalesce(row.content, '')
    RETURN count(review)
    """),
    params = {'file': url_node_review_2}
)

[<Record count(review)=12054>]

In [15]:
# df_node_user
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE(user:User {id: toInteger(row.id), name: row.name})
    RETURN count(user)
    """),
    params = {'file': url_node_user}
)

[<Record count(user)=21315>]

In [16]:
# df_node_district
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE(district:District {id: toInteger(row.id)})
    ON CREATE SET district.name = row.name
    RETURN count(district)
    """),
    params = {'file': url_node_district}
)

[<Record count(district)=22>]

In [17]:
# df_node_ward
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MERGE (ward:Ward {code: row.ward_code})
    ON CREATE SET
        ward.name = row.name,
        ward.province_code = row.province_code
    WITH ward, row
    WHERE row.district_id IS NOT NULL
    MATCH (district:District {id: toInteger(row.district_id)})
    MERGE (ward)-[r:BELONGS_TO]->(district)
    RETURN count(ward)
    """),
    params = {'file': url_node_ward}
)

[<Record count(ward)=280>]

# Tải các Relationship (Quan hệ)

In [18]:
# df_poi_belongsto_category
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MATCH (poi:Poi {id: toInteger(row.poi_id)})
    MATCH (category:Category {id: toInteger(row.category_id)})
    MERGE (poi)-[r:BELONGS_TO]->(category)
    RETURN count(r) AS BELONGS_TO_count
    """),
    params = {'file': url_poi_belongsto_category}
)

[<Record BELONGS_TO_count=3370>]

In [19]:
# df_poi_locatedat_region
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MATCH (poi:Poi {id: toInteger(row.poi_id)})
    MATCH (region:Region {id: toInteger(row.region_id)})
    MERGE (poi)-[r:LOCATED_AT]->(region)
    RETURN count(r) AS LOCATED_AT_count
    """),
    params = {'file': url_poi_locatedat_region}
)

[<Record LOCATED_AT_count=3326>]

In [ ]:
# df_poi_nearby_poi
run(driver, "CREATE POINT INDEX poi_location_index IF NOT EXISTS FOR (p:Poi) ON (p.location)")
run(driver, textwrap.dedent("""\
    MATCH (p1:Poi)
    WHERE p1.location IS NOT NULL
    CALL (p1) {
        MATCH (p2:Poi)
        WHERE p2.location IS NOT NULL 
          AND p1.id < p2.id
        WITH p1, p2, point.distance(p1.location, p2.location) AS dist_m
        WHERE dist_m <= 1500
        WITH p1, p2, round(dist_m / 1000.0, 2) AS dist_km
        MERGE (p1)-[r1:NEARBY]->(p2) ON CREATE SET r1.distance_km = dist_km
        MERGE (p2)-[r2:NEARBY]->(p1) ON CREATE SET r2.distance_km = dist_km
        RETURN count(r1) AS count_r1
    } IN TRANSACTIONS OF 100 ROWS
    RETURN sum(count_r1) * 2 AS NEARBY_relations_created
"""))

[<Record NEARBY_relations_created=1563580>]

In [22]:
# df_user_from_origin
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MATCH (user:User {id: toInteger(row.user_id)})
    MATCH (origin:Origin {id: toInteger(row.origin_id)})
    MERGE (user)-[r:FROM]->(origin)
    RETURN count(r) AS FROM_count
    """),
    params = {'file': url_user_from_origin}
)

[<Record FROM_count=9557>]

In [23]:
# df_poi_locatedin_ward
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MATCH (poi:Poi {id: toInteger(row.poi_id)})
    MATCH (ward:Ward {code: row.ward_code})
    MERGE (poi)-[r:LOCATED_IN]->(ward)
    RETURN count(r) AS LOCATED_IN_count
    """),
    params = {'file': url_poi_locatedin_ward}
)

[<Record LOCATED_IN_count=1185>]

In [24]:
# df_ward_mergedto_ward
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    MATCH (old_w:Ward {code: row.old_ward_code})
    MATCH (new_w:Ward {code: row.new_ward_code})
    MERGE (old_w)-[r:MERGED_TO]->(new_w)
    RETURN count(r) AS MERGED_TO_count
    """),
    params = {'file': url_ward_mergedto_ward}
)

[<Record MERGED_TO_count=275>]

In [25]:
# df_user_reviewed_poi
run(driver, textwrap.dedent("""\
    LOAD CSV WITH HEADERS FROM $file AS row
    CALL (row) {
        MATCH (user:User {id: toInteger(row.user_id)})
        MATCH (review:Review {id: toInteger(row.review_id)})
        MATCH (poi:Poi {id: toInteger(row.poi_id)})
        MERGE (user)-[w:WROTE]->(review)
        MERGE (review)-[rated:RATED]->(poi)
        MERGE (user)-[reviewed:REVIEWED ]->(poi)
        ON CREATE SET reviewed.rating = review.rating
        RETURN count(w) AS WROTE_count, count(rated) AS RATED_count, count(reviewed) AS REVIEWED_count
    } IN TRANSACTIONS
    RETURN SUM(WROTE_count) AS total_WROTE_count, SUM(RATED_count) AS total_RATED_count, SUM(REVIEWED_count) AS total_REVIEWED_count
    """),
    params = {'file': url_user_reviewed_poi}
)

[<Record total_WROTE_count=24108 total_RATED_count=24108 total_REVIEWED_count=24108>]

# Đóng kết nối driver

In [26]:
driver.close()